<a href="https://colab.research.google.com/github/jacobhroutzong/Decision-Models/blob/main/Decision_Model_Kev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
%cd /content

# Uninstall all incompatible/mismatched pre-installed torch auxiliary packages
!pip uninstall -y torchvision torchaudio torchao -q

# Clean install of Kev
!rm -rf /content/kev
!git clone https://github.com/jaredpalmer/kev.git
%cd /content/kev

!pip install -q uv
!uv pip install --system -e ".[serve]"
!pip install -q typesafe-sdk

/content
Cloning into 'kev'...
remote: Enumerating objects: 11213, done.
remote: Counting objects: 100% (2487/2487), done.
remote: Compressing objects: 100% (612/612), done.
remote: Total 11213 (delta 2110), reused 1911 (delta 1849), pack-reused 8726 (from 2)
Receiving objects: 100% (11213/11213), 167.79 MiB | 36.09 MiB/s, done.
Resolving deltas: 100% (6990/6990), done.
Updating files: 100% (3051/3051), done.
/content/kev
Using Python 3.13.15 environment at: /usr
Resolved 92 packages in 129ms
Prepared 1 package in 832ms
Uninstalled 1 package in 0.56ms
Installed 1 package in 0.91ms
 ~ kev==0.1.0 (from file:///content/kev)


In [23]:
import subprocess
import time
import socket

MODEL_ID = "jaredpalmer/kev-4b"
PORT = 8009

# Clean up any leftover processes using port 8009
!fuser -k 8009/tcp 2>/dev/null || true

print(f"Launching Kev server ({MODEL_ID}) on port {PORT}...")

# Launch with unbuffered stdout so all logs print immediately
process = subprocess.Popen(
    ["python", "-u", "-m", "kev.serve", "--run", MODEL_ID, "--port", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

server_ready = False
start_time = time.time()

# Stream logs in real-time until the server is ready
while time.time() - start_time < 300:
    line = process.stdout.readline()
    if line:
        print(line, end="")

    if process.poll() is not None:
        remaining = process.stdout.read()
        if remaining:
            print(remaining)
        print(f"\n[ERROR] Server process died with code {process.returncode}")
        break

    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=0.5):
            server_ready = True
            print(f"\n Server is LIVE and listening on http://127.0.0.1:{PORT}!")
            break
    except (ConnectionRefusedError, OSError):
        pass

if not server_ready and process.poll() is None:
    print("\nServer is still downloading weights or initializing...")

 23944Launching Kev server (jaredpalmer/kev-4b) on port 8009...

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 1722.13it/s]

Loading weights: 100%|██████████| 426/426 [00:00<00:00, 5599.99it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
fused Qwen3.5 kernels off: install the flash-linear-attention version kev/fused_qwen35.py pins (FLA_VERSION) to turn them on
serving jaredpalmer/kev-4b (/root/.cache/huggingface/hub/models--jaredpalmer--kev-4b/snapshots/6cfce5c2fa4b4bd64026336ab649c5ca78857d52) on cuda via torch (bfloat16) 127.0.0.1:8009; states over 65,536 tokens refused (

In [24]:
import sys
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score

console = Console()

# 1. Connect to local Kev server
client = TypeSafeClient(
    api_key="local",
    base_url="http://127.0.0.1:8009",
    model="kev-latest"
)

ticket_text = (
    "Hi support, I was billed twice for my annual subscription renewal on October 3rd. "
    "I tried calling your hotline but was on hold for 45 minutes before being disconnected. "
    "Please reverse the $120 charge immediately or I will contact my bank to issue a chargeback."
)

# 2. Forward pass inference
response = client.system_one(
    state=ticket_text,
    questions={
        "department": Choice(
            instructions="Which department should handle this request?",
            criteria={
                "billing": "Inquiries regarding charges, payments, double billing, or refunds.",
                "technical_support": "Technical issues, app crashes, bug reports, or error codes.",
                "account_cancellation": "Direct requests to close, delete, or cancel an account.",
                "general_inquiry": "General feedback, feature questions, or sales inquiries.",
            }
        ),
        "churn_risk": Noul(
            instructions="Does the user threaten cancellation, chargeback, or leaving the service?"
        ),
        "urgency": Score(
            instructions="Rate the urgency of this ticket.",
            criteria=[
                "Low: General inquiry with no financial impact.",
                "Medium: Minor inconvenience or standard follow-up.",
                "High: Customer upset or ongoing service interruption.",
                "Critical: Immediate threat of legal action, chargeback, or revenue loss."
            ]
        ),
    },
)

# 3. Helper to draw probability bars
def make_bar(prob: float, width: int = 15) -> str:
    filled = int(round(prob * width))
    empty = width - filled
    return f"[{'█' * filled}{'░' * empty}]"

# --- RENDER DASHBOARD ---
answers = response.answers

# Top-line Metrics Summary Table
summary_table = Table(title="Decision Overview", box=None, header_style="bold cyan")
summary_table.add_column("Task", style="bold")
summary_table.add_column("Top Decision", style="bold green")
summary_table.add_column("Confidence / Value")
summary_table.add_column("Distribution Preview")

# A. Choice (Department)
dept = answers["department"]
dept_bar = make_bar(dept.confidence)
summary_table.add_row(
    "Department",
    dept.choice.upper(),
    f"{dept.confidence * 100:.1f}%",
    f"[green]{dept_bar}[/green]"
)

# B. Noul (Churn Risk)
churn = answers["churn_risk"]
churn_status = "CRITICAL RISK" if churn.noul > 0.7 else ("MODERATE RISK" if churn.noul > 0.4 else "LOW RISK")
churn_color = "red" if churn.noul > 0.7 else ("yellow" if churn.noul > 0.4 else "green")
churn_bar = make_bar(churn.noul)
summary_table.add_row(
    "Churn / Dispute",
    f"[{churn_color}]{churn_status}[/{churn_color}]",
    f"{churn.noul * 100:.1f}% prob",
    f"[{churn_color}]{churn_bar}[/{churn_color}]"
)

# C. Score (Urgency)
urg = answers["urgency"]
urg_idx = int(round(urg.score))
urg_label = urg.legend.get(urg_idx, urg.legend.get(str(urg_idx), f"Level {urg_idx}")).split(":")[0]
urg_bar = make_bar(urg.confidence)
summary_table.add_row(
    "Urgency Level",
    f"[bold red]{urg_label.upper()} ({urg.score:.2f}/3.0)[/bold red]",
    f"{urg.confidence * 100:.1f}% certainty",
    f"[red]{urg_bar}[/red]"
)

# Context Box
state_panel = Panel(
    Text(ticket_text, style="italic dim"),
    title="[bold yellow]Evaluated Context / State[/bold yellow]",
    border_style="dim"
)

# Detailed Breakdown Table
breakdown_table = Table(title="Calibrated Probability Details", expand=True)
breakdown_table.add_column("Task / Option", style="bold")
breakdown_table.add_column("Probability", justify="right")
breakdown_table.add_column("Visual Distribution", ratio=1)

# Populate Choice Probabilities
breakdown_table.add_section()
for opt, prob in sorted(dept.probabilities.items(), key=lambda x: x[1], reverse=True):
    color = "green" if opt == dept.choice else "white"
    breakdown_table.add_row(
        f"Dept: {opt}",
        f"{prob * 100:.2f}%",
        f"[{color}]{make_bar(prob, 25)}[/{color}]"
    )

# Populate Urgency Probabilities
breakdown_table.add_section()
for lvl, prob in urg.probabilities.items():
    desc = urg.legend.get(int(lvl), urg.legend.get(str(lvl), f"Level {lvl}")).split(":")[0]
    color = "red" if prob > 0.5 else ("yellow" if prob > 0.2 else "dim")
    breakdown_table.add_row(
        f"Urgency: {desc}",
        f"{prob * 100:.2f}%",
        f"[{color}]{make_bar(prob, 25)}[/{color}]"
    )

# Display everything
console.print(state_panel)
console.print(summary_table)
console.print(breakdown_table)

╭─────────────────────────────────────────── Evaluated Context / State ───────────────────────────────────────────╮
│ Hi support, I was billed twice for my annual subscription renewal on October 3rd. I tried calling your hotline  │
│ but was on hold for 45 minutes before being disconnected. Please reverse the $120 charge immediately or I will  │
│ contact my bank to issue a chargeback.                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                               Decision Overview                                
 Task             Top Decision         Confidence / Value  Distribution Preview 
 Department       BILLING              96.0%               [██████████████░]    
 Churn / Dispute  CRITICAL RISK        83.6% prob          [█████████████░░]    
 Urgency Level    CRITICAL (2.59/3.0)  59.0% certainty     [█████████░░░░░░]

                                          Calibrated Probability Details                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Task / Option              ┃ Probability ┃ Visual Distribution                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Dept: billing              │      96.99% │ [████████████████████████░]                                          │
│ Dept: general_inquiry      │       1.33% │ [░░░░░░░░░░░░░░░░░░░░░░░░░]                                          │
│ Dept: account_cancellation │       0.94% │ [░░░░░░░░░░░░░░░░░░░░░░░░░]                                          │
│ Dept: technical_support    │       0.74% │ [░░░░░░░░░░░░░░░░░░░░░░░░░]                                          │
├────────────────────────────┼─────────────┼──────────────────────────────────────────────────────────────────────┤
│ Urgency: Low               │       2.06% │ [█░░░░░░░░░░░░░░░░░░░░░░░░]                                          │
│ Urgency: Medium            │       4.67% │ [█░░░░░░░░░░░░░░░░░░░░░░░░]                                          │
│ Urgency: High              │      25.47% │ [██████░░░░░░░░░░░░░░░░░░░]                                          │
│ Urgency: Critical          │      67.79% │ [█████████████████░░░░░░░░]                                          │
└────────────────────────────┴─────────────┴──────────────────────────────────────────────────────────────────────┘